# Basic RAG Pipeline

This notebook demonstrates a complete Retrieval-Augmented Generation (RAG)
pipeline using the built-in components provided by RAG Framework.

The example does not require an API key or an external language model.

We will walk through the following steps:

1. Install the framework
2. Load a document
3. Split the document into chunks
4. Generate embeddings
5. Ingest the chunks into the retriever
6. Query the RAG pipeline
7. Inspect the retrieved results

The built-in `RandomEmbedder` and `EchoGenerator` are used so that the
example can run without external API credentials.

## 1. Installation

Install the framework in editable mode with its development dependencies.

When working from a cloned repository, installing with `-e` allows changes
to the source code to be reflected immediately without reinstalling the
package.

In [ ]:
import sys
import tempfile
from pathlib import Path

from ragframework.config import RAGConfig
from ragframework.document.chunkers import FixedSizeChunker
from ragframework.document.loaders import TextFileLoader
from ragframework.embeddings.random_embedder import RandomEmbedder
from ragframework.generator.echo_generator import EchoGenerator
from ragframework.pipeline.rag import RAGPipeline
from ragframework.retriever.in_memory import InMemoryRetriever

## 3. Create a Sample Document

To keep this example self-contained, we create a small text document
containing an explanation of RAG.

Using a temporary file means the example does not depend on any external
dataset or file.

In [ ]:
SAMPLE_TEXT = """\
Retrieval-Augmented Generation (RAG) is a technique that combines information
retrieval with text generation. Rather than relying solely on knowledge baked
into a language model's parameters, RAG retrieves relevant documents from an
external corpus and injects them as context before generating an answer.

This approach offers several advantages: it keeps the knowledge base up-to-date
without retraining, reduces hallucinations by grounding responses in retrieved
evidence, and allows fine-grained control over what sources are consulted.

A typical RAG pipeline consists of four stages:
1. Document ingestion — load and chunk source documents.
2. Embedding — convert chunks to dense vectors.
3. Retrieval — find the chunks most similar to a query.
4. Generation — produce an answer conditioned on the retrieved context.

RAG Framework provides clean abstractions for each stage so you can swap in
your preferred implementations (OpenAI, HuggingFace, ChromaDB, FAISS, ...) 
without changing the pipeline logic.
"""

with tempfile.NamedTemporaryFile(
    mode="w",
    suffix=".txt",
    delete=False,
    encoding="utf-8",
) as f:
    f.write(SAMPLE_TEXT)
    tmp_path = f.name

print(f"Sample document: {tmp_path}")

## 4. Load the Document

The `TextFileLoader` reads the text file and converts it into the framework's
`Document` representation.

A document contains its text along with metadata such as its identifier.

In [ ]:
loader = TextFileLoader()

documents = loader.load(tmp_path)

print(f"Loaded {len(documents)} document(s)")
print(documents[0].content[:300])

## 5. Chunk the Document

Long documents are usually split into smaller pieces before embedding.

Here we use `FixedSizeChunker` with a chunk size of 200 characters and an
overlap of 40 characters. The overlap helps preserve context between
neighboring chunks.

In [ ]:
chunker = FixedSizeChunker(
    chunk_size=200,
    chunk_overlap=40,
)

chunks = chunker.chunk(documents[0])

print(f"Created {len(chunks)} chunks")

## 6. Generate Embeddings

Embeddings represent text as numerical vectors.

For this example we use the built-in `RandomEmbedder`. It generates
deterministic random vectors and is useful for demonstrating the framework
without requiring an external embedding API.

For meaningful semantic retrieval, a real embedding implementation such as
OpenAI or Hugging Face should be used.

In [ ]:
embedder = RandomEmbedder(dim=64, seed=42)

embeddings = embedder.embed([chunk.content for chunk in chunks])

print(f"Generated {len(embeddings)} embeddings")
print(f"Embedding dimension: {len(embeddings[0])}")

## 7. Ingest the Documents

The RAG pipeline combines loading, chunking, embedding, and retrieval.

We configure the pipeline using the framework's built-in components.

In [ ]:
pipeline = RAGPipeline(
    loader=TextFileLoader(),
    chunker=FixedSizeChunker(
        chunk_size=200,
        chunk_overlap=40,
    ),
    embedder=RandomEmbedder(
        dim=64,
        seed=42,
    ),
    retriever=InMemoryRetriever(),
    generator=EchoGenerator(),
    config=RAGConfig(top_k=3),
)
n_chunks = pipeline.ingest(tmp_path)

print(f"Ingested {n_chunks} chunks")

## 8. Query the RAG Pipeline

Now we can ask a question about the information contained in our document.

The pipeline embeds the query, retrieves the configured number of relevant
chunks, and passes those chunks to the generator.

In [ ]:
query = "What are the stages of a RAG pipeline?"

response = pipeline.query(query)

print("Query:")
print(query)

print("\nAnswer:")
print(response.answer)

## 9. Inspect Retrieved Results

The response also contains the source chunks retrieved for the query.

Inspecting these chunks is useful for understanding which pieces of the
knowledge base were provided as context to the generator.

In [ ]:
print(f"Number of source chunks: {len(response.source_chunks)}")

for i, chunk in enumerate(response.source_chunks, start=1):
    print(f"\n--- Source Chunk {i} ---")
    print(chunk.content)

## 10. Cleanup

The sample document was created as a temporary file, so we remove it after
the example is complete.

In [ ]:
Path(tmp_path).unlink(missing_ok=True)

print("Temporary file removed.")